# Retail Intelligence Platform - SQL Executive KPI Analysis

This notebook connects to PostgreSQL and builds the executive KPI layer for the Retail Intelligence Platform.

## Objectives
- connect to PostgreSQL
- create reusable order-level analytical views
- calculate executive KPIs
- analyze monthly revenue trend
- analyze order status distribution
- export clean KPI outputs to `sql/outputs/`

## PostgreSQL tables used
- `olist_orders`
- `olist_order_items`
- `olist_order_payments`
- `olist_order_reviews`
- `olist_customers`
- `olist_products`
- `olist_sellers`
- `product_category_name_translation`
- `olist_geolocation`

In [5]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

# --------------------------------------------------
# PostgreSQL connection
# --------------------------------------------------
DB_USER = "postgres"
DB_PASSWORD = quote_plus("1234")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail_intelligence"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("PostgreSQL engine created successfully.")

PostgreSQL engine created successfully.


## 1. Quick Connection Test
Run a simple query to confirm that the notebook can read from PostgreSQL.

In [6]:
test_query = """
SELECT COUNT(*) AS orders_count
FROM olist_orders;
"""

df_test = pd.read_sql(test_query, engine)
df_test

,orders_count
0,99441


## 2. Create Reusable Analytical Views

To avoid KPI duplication and messy joins, we first create a small set of reusable views:

### Views created
- `vw_orders_enriched` -> order + customer information
- `vw_order_revenue` -> order-level revenue and freight summary
- `vw_order_payments` -> order-level payment summary
- `vw_order_reviews` -> order-level review summary
- `vw_sales_fact` -> final one-row-per-order analytical fact view

In [7]:
create_views_sql = """
-- ------------------------------------------------------------
-- View 1: vw_orders_enriched
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_orders_enriched;

CREATE VIEW vw_orders_enriched AS
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.order_purchase_timestamp::date AS order_date,
    DATE_TRUNC('month', o.order_purchase_timestamp)::date AS order_month
FROM olist_orders o
LEFT JOIN olist_customers c
    ON o.customer_id = c.customer_id;


-- ------------------------------------------------------------
-- View 2: vw_order_revenue
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_order_revenue;

CREATE VIEW vw_order_revenue AS
SELECT
    oi.order_id,
    SUM(oi.price) AS item_revenue,
    SUM(oi.freight_value) AS freight_revenue,
    SUM(oi.price + oi.freight_value) AS gross_order_value,
    COUNT(*) AS order_item_count,
    COUNT(DISTINCT oi.product_id) AS distinct_products,
    COUNT(DISTINCT oi.seller_id) AS distinct_sellers
FROM olist_order_items oi
GROUP BY oi.order_id;


-- ------------------------------------------------------------
-- View 3: vw_order_payments
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_order_payments;

CREATE VIEW vw_order_payments AS
SELECT
    p.order_id,
    SUM(p.payment_value) AS total_payment_value,
    COUNT(*) AS payment_rows,
    MAX(p.payment_installments) AS max_installments
FROM olist_order_payments p
GROUP BY p.order_id;


-- ------------------------------------------------------------
-- View 4: vw_order_reviews
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_order_reviews;

CREATE VIEW vw_order_reviews AS
SELECT
    r.order_id,
    AVG(r.review_score) AS avg_review_score,
    COUNT(*) AS review_count
FROM olist_order_reviews r
GROUP BY r.order_id;


-- ------------------------------------------------------------
-- View 5: vw_sales_fact
-- ONE ROW PER ORDER analytical fact table
-- ------------------------------------------------------------
DROP VIEW IF EXISTS vw_sales_fact;

CREATE VIEW vw_sales_fact AS
SELECT
    oe.order_id,
    oe.customer_id,
    oe.customer_unique_id,
    oe.customer_city,
    oe.customer_state,
    oe.order_status,
    oe.order_purchase_timestamp,
    oe.order_approved_at,
    oe.order_delivered_carrier_date,
    oe.order_delivered_customer_date,
    oe.order_estimated_delivery_date,
    oe.order_date,
    oe.order_month,

    COALESCE(orv.item_revenue, 0) AS item_revenue,
    COALESCE(orv.freight_revenue, 0) AS freight_revenue,
    COALESCE(orv.gross_order_value, 0) AS gross_order_value,
    COALESCE(orv.order_item_count, 0) AS order_item_count,
    COALESCE(orv.distinct_products, 0) AS distinct_products,
    COALESCE(orv.distinct_sellers, 0) AS distinct_sellers,

    COALESCE(op.total_payment_value, 0) AS total_payment_value,
    COALESCE(op.payment_rows, 0) AS payment_rows,
    COALESCE(op.max_installments, 0) AS max_installments,

    rv.avg_review_score,
    COALESCE(rv.review_count, 0) AS review_count

FROM vw_orders_enriched oe
LEFT JOIN vw_order_revenue orv
    ON oe.order_id = orv.order_id
LEFT JOIN vw_order_payments op
    ON oe.order_id = op.order_id
LEFT JOIN vw_order_reviews rv
    ON oe.order_id = rv.order_id;
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_views_sql)

print("Views created successfully.")

Views created successfully.


## 3. Validate the analytical fact view
The final view `vw_sales_fact` should contain one row per order and act as the base for executive KPI calculations.

In [8]:
fact_check_query = """
SELECT COUNT(*) AS total_rows
FROM vw_sales_fact;
"""

df_fact_check = pd.read_sql(fact_check_query, engine)
df_fact_check

,total_rows
0,99441


In [9]:
fact_preview_query = """
SELECT *
FROM vw_sales_fact
LIMIT 10;
"""

df_fact_preview = pd.read_sql(fact_preview_query, engine)
df_fact_preview

,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,freight_revenue,gross_order_value,order_item_count,distinct_products,distinct_sellers,total_payment_value,payment_rows,max_installments,avg_review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,...,13.29,72.19,1,1,1,72.19,1,2,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,...,19.93,259.83,1,1,1,259.83,1,3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,...,17.87,216.87,1,1,1,216.87,1,5,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,...,12.79,25.78,1,1,1,25.78,1,2,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,...,18.14,218.04,1,1,1,218.04,1,3,5.0,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,816cbea969fe5b689b39cfc97a506742,85c835d128beae5b4ce8602c491bf385,uberaba,MG,delivered,2017-05-15 21:42:34,2017-05-17 03:55:27,2017-05-17 11:05:55,2017-05-22 13:44:35,...,12.69,34.59,1,1,1,34.59,1,1,4.0,1
6,00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,635d9ac1680f03288e72ada3a1035803,guararapes,SP,delivered,2017-12-10 11:53:48,2017-12-10 12:10:31,2017-12-12 01:07:48,2017-12-18 22:03:38,...,11.85,31.75,1,1,1,31.75,1,1,4.0,1
7,000576fe39319847cbb9d288c5617fa6,9ed5e522dd9dd85b4af4a077526d8117,fda4476abb6307ab3c415b7e6d026526,praia grande,SP,delivered,2018-07-04 12:08:27,2018-07-05 16:35:48,2018-07-05 12:15:00,2018-07-09 14:04:07,...,70.75,880.75,1,1,1,880.75,1,10,5.0,1
8,0005a1a1728c9d785b8e2b08b904576c,16150771dfd4776261284213b89c304e,639d23421f5517f69d0c3d6e6564cf0e,santos,SP,delivered,2018-03-19 18:40:33,2018-03-20 18:35:21,2018-03-28 00:37:42,2018-03-29 18:17:31,...,11.65,157.60,1,1,1,157.60,1,3,1.0,1
9,0005f50442cb953dcd1d21e1fb923495,351d3cb2cee3c7fd0af6616c82df21d3,0782c41380992a5a533489063df0eef6,jandira,SP,delivered,2018-07-02 13:59:39,2018-07-02 14:10:56,2018-07-03 14:25:00,2018-07-04 17:28:31,...,11.40,65.39,1,1,1,65.39,1,1,4.0,1


## 4. Executive KPI Summary

This section calculates the main executive KPIs for the retail business:
- total orders
- total customers
- total revenue
- average order value
- average review score
- delivered orders
- cancelled orders
- delivered order rate
- total freight cost
- freight cost as a percentage of revenue

In [10]:
executive_kpi_query = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_customers,
    ROUND(SUM(item_revenue)::numeric, 2) AS total_revenue,
    ROUND(AVG(item_revenue)::numeric, 2) AS average_order_value,
    ROUND(AVG(avg_review_score)::numeric, 2) AS average_review_score,
    COUNT(DISTINCT CASE WHEN order_status = 'delivered' THEN order_id END) AS delivered_orders,
    COUNT(DISTINCT CASE WHEN order_status = 'canceled' THEN order_id END) AS cancelled_orders,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN order_status = 'delivered' THEN order_id END)
        / NULLIF(COUNT(DISTINCT order_id), 0),
        2
    ) AS delivered_order_rate_pct,
    ROUND(SUM(freight_revenue)::numeric, 2) AS total_freight_cost,
    ROUND(
        100.0 * SUM(freight_revenue) / NULLIF(SUM(item_revenue), 0),
        2
    ) AS freight_pct_of_revenue
FROM vw_sales_fact;
"""

df_executive_kpis = pd.read_sql(executive_kpi_query, engine)
df_executive_kpis

,total_orders,total_customers,total_revenue,average_order_value,average_review_score,delivered_orders,cancelled_orders,delivered_order_rate_pct,total_freight_cost,freight_pct_of_revenue
0,99441,96096,13591643.7,136.68,4.09,96478,625,97.02,2251909.54,16.57


## 5. Monthly Revenue Trend
This section summarizes monthly orders, revenue, freight cost, and average order value.

In [11]:
monthly_revenue_query = """
SELECT
    order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(item_revenue)::numeric, 2) AS total_revenue,
    ROUND(SUM(freight_revenue)::numeric, 2) AS total_freight_cost,
    ROUND(AVG(item_revenue)::numeric, 2) AS average_order_value
FROM vw_sales_fact
GROUP BY order_month
ORDER BY order_month;
"""

df_monthly_revenue = pd.read_sql(monthly_revenue_query, engine)
df_monthly_revenue

,order_month,total_orders,total_revenue,total_freight_cost,average_order_value
0,2016-09-01,4,267.36,87.39,66.84
1,2016-10-01,324,49507.66,7301.18,152.80
2,2016-12-01,1,10.90,8.72,10.90
3,2017-01-01,800,120312.87,16875.62,150.39
4,2017-02-01,1780,247303.02,38977.60,138.93
5,2017-03-01,2682,374344.30,57704.29,139.58
6,2017-04-01,2404,359927.23,52495.01,149.72
7,2017-05-01,3700,506071.14,80119.81,136.78
8,2017-06-01,3245,433038.60,69924.44,133.45
9,2017-07-01,4026,498031.48,86940.14,123.70


## 6. Monthly Revenue Growth
Calculate month-over-month revenue change to understand growth and decline patterns.

In [12]:
monthly_growth_query = """
WITH monthly_revenue AS (
    SELECT
        order_month,
        SUM(item_revenue) AS total_revenue
    FROM vw_sales_fact
    GROUP BY order_month
)
SELECT
    order_month,
    ROUND(total_revenue::numeric, 2) AS total_revenue,
    ROUND(
        LAG(total_revenue) OVER (ORDER BY order_month)::numeric,
        2
    ) AS previous_month_revenue,
    ROUND(
        (
            (total_revenue - LAG(total_revenue) OVER (ORDER BY order_month))
            / NULLIF(LAG(total_revenue) OVER (ORDER BY order_month), 0)
        ) * 100,
        2
    ) AS revenue_growth_pct
FROM monthly_revenue
ORDER BY order_month;
"""

df_monthly_growth = pd.read_sql(monthly_growth_query, engine)
df_monthly_growth

,order_month,total_revenue,previous_month_revenue,revenue_growth_pct
0,2016-09-01,267.36,NaN,NaN
1,2016-10-01,49507.66,267.36,18417.23
2,2016-12-01,10.90,49507.66,-99.98
3,2017-01-01,120312.87,10.90,1103687.80
4,2017-02-01,247303.02,120312.87,105.55
5,2017-03-01,374344.30,247303.02,51.37
6,2017-04-01,359927.23,374344.30,-3.85
7,2017-05-01,506071.14,359927.23,40.60
8,2017-06-01,433038.60,506071.14,-14.43
9,2017-07-01,498031.48,433038.60,15.01


## 7. Order Status Distribution
This shows how orders are distributed across delivery states such as delivered, shipped, canceled, etc.

In [13]:
order_status_query = """
SELECT
    order_status,
    COUNT(DISTINCT order_id) AS orders_count
FROM vw_sales_fact
GROUP BY order_status
ORDER BY orders_count DESC;
"""

df_order_status = pd.read_sql(order_status_query, engine)
df_order_status

,order_status,orders_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 8. Executive KPI Health Labels
Simple KPI interpretation layer to make the executive summary more business-friendly.

In [14]:
kpi_status_query = """
WITH kpis AS (
    SELECT
        ROUND(AVG(avg_review_score)::numeric, 2) AS average_review_score,
        ROUND(
            100.0 * COUNT(DISTINCT CASE WHEN order_status = 'delivered' THEN order_id END)
            / NULLIF(COUNT(DISTINCT order_id), 0),
            2
        ) AS delivered_order_rate_pct,
        ROUND(
            100.0 * SUM(freight_revenue) / NULLIF(SUM(item_revenue), 0),
            2
        ) AS freight_pct_of_revenue
    FROM vw_sales_fact
)
SELECT
    average_review_score,
    delivered_order_rate_pct,
    freight_pct_of_revenue,

    CASE
        WHEN average_review_score >= 4.0 THEN 'Good'
        WHEN average_review_score >= 3.5 THEN 'Average'
        ELSE 'Needs Attention'
    END AS review_kpi_status,

    CASE
        WHEN delivered_order_rate_pct >= 95 THEN 'Healthy'
        WHEN delivered_order_rate_pct >= 90 THEN 'Moderate'
        ELSE 'Needs Attention'
    END AS delivery_kpi_status,

    CASE
        WHEN freight_pct_of_revenue <= 15 THEN 'Healthy'
        WHEN freight_pct_of_revenue <= 20 THEN 'Moderate'
        ELSE 'High Freight Cost'
    END AS freight_kpi_status
FROM kpis;
"""

df_kpi_status = pd.read_sql(kpi_status_query, engine)
df_kpi_status

,average_review_score,delivered_order_rate_pct,freight_pct_of_revenue,review_kpi_status,delivery_kpi_status,freight_kpi_status
0,4.09,97.02,16.57,Good,Healthy,Moderate


## 9. Export outputs
Save the executive SQL outputs into the project `sql/outputs/` folder for reuse in reporting and documentation.

In [15]:
OUTPUT_DIR = Path(r"C:\Users\divya\Retail-Intelligence-Platform\sql\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_executive_kpis.to_csv(OUTPUT_DIR / "executive_kpis.csv", index=False)
df_monthly_revenue.to_csv(OUTPUT_DIR / "monthly_revenue.csv", index=False)
df_monthly_growth.to_csv(OUTPUT_DIR / "monthly_revenue_growth.csv", index=False)
df_order_status.to_csv(OUTPUT_DIR / "order_status_summary.csv", index=False)
df_kpi_status.to_csv(OUTPUT_DIR / "executive_kpi_status.csv", index=False)

print("Executive SQL outputs exported successfully.")
print(f"Saved to: {OUTPUT_DIR}")

Executive SQL outputs exported successfully.
Saved to: C:\Users\divya\Retail-Intelligence-Platform\sql\outputs


## 10. Summary

This notebook created the SQL executive KPI layer for the Retail Intelligence Platform.

### Outputs generated
- `executive_kpis.csv`
- `monthly_revenue.csv`
- `monthly_revenue_growth.csv`
- `order_status_summary.csv`
- `executive_kpi_status.csv`

### Next notebook
`02_sql_customer_analysis.ipynb`